<a href="https://colab.research.google.com/github/crialejo24/DOWNSCALING/blob/main/Entrenamiento_SWIN_IR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#!pip install timm
#!pip install einops

Mounted at /content/drive


# Nuevo entrenamiento

## Cargar repositorio oficial

In [ ]:
#REPOSITORIO ORIGNAL
!git clone https://github.com/cszn/KAIR.git

%cd KAIR
!pip install -r requirement.txt

## Reemplazar archivos modificados

In [ ]:
# Editar archivo para seleccionar las capaz entrenables de la arquitectura
!cp /content/drive/MyDrive/DOWNSCALING_2/modificables_andres/model_plain.py /content/KAIR/models/

# Editar archivo para ignorar advertencias en la lectura de archivos .tiff
!cp /content/drive/MyDrive/DOWNSCALING_2/modificables_andres/utils_image.py /content/KAIR/utils/

## Cargar modelo preentrenado

In [ ]:
!wget https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/001_classicalSR_DF2K_s64w8_SwinIR-M_x3.pth -P model_zoo/

## Crear carpetas del dataset

In [ ]:
!ls model_zoo
!mkdir -p trainsets/trainH
!mkdir -p trainsets/trainL
!mkdir -p testsets/testH
!mkdir -p testsets/testL

001_classicalSR_DF2K_s64w8_SwinIR-M_x3.pth  dncnn_25.pth  README.md


## Subir imagenes del dataset

In [ ]:
!cp -r /content/drive/MyDrive/DOWNSCALING_2/train_GeoR/HR_192_mod/* trainsets/trainH/
!cp -r /content/drive/MyDrive/DOWNSCALING_2/train_GeoR/LR_64_modx3/* trainsets/trainL/

!cp -r /content/drive/MyDrive/DOWNSCALING_2/test_GeoR/HR_192_mod/* testsets/testH/
!cp -r /content/drive/MyDrive/DOWNSCALING_2/test_GeoR/LR_64_modx3/* testsets/testL/

## Configuración del entrenamiento

Es importante asignar en "root" una ruta externa al Colab para guardar el avance del entrenamiento y evitar perderlo si hay un cierre o reinicio de sesión del Colab

In [ ]:
import json
import os

os.makedirs("options/swinir", exist_ok=True)

config = {
  "task": "swinir_sr_x3_finetune"     #classical image sr for x2/x3/x4/x8. root/task/images-models-options
  , "model": "plain" # "plain" | "plain2" if two inputs
  , "gpu_ids": [0]
  , "dist": False

  , "scale": 3       # 2 | 3 | 4 | 8
  , "n_channels": 3  # broadcast to "datasets", 1 for grayscale, 3 for color

  , "path": {
    "root": "/content/drive/MyDrive/DOWNSCALING_2/Repositorio_TRAIN_2/KAIR_UPSAMPLE_LAYER5/superresolution" # Ruta donde se guarda el avance del entrenamiento
    , "pretrained_netG": "model_zoo/001_classicalSR_DF2K_s64w8_SwinIR-M_x3.pth"      # path of pretrained model. We fine-tune X3/X4/X8 models from X2 model, so that `G_optimizer_lr` and `G_scheduler_milestones` can be halved to save time.
    , "pretrained_netE": None      # path of pretrained model
  }

  , "datasets": {
    "train": {
      "name": "train_dataset"           #// just name
      , "dataset_type": "sr"         #// "dncnn" | "dnpatch" | "fdncnn" | "ffdnet" | "sr" | "srmd" | "dpsr" | "plain" | "plainpatch" | "jpeg"
      , "dataroot_H": "trainsets/trainH" #// path of H training dataset. DIV2K (800 training images)
      , "dataroot_L": "trainsets/trainL"  #            // path of L training dataset

      , "H_size": 192                   #// 96/144|192/384 | 128/192/256/512. LR patch size is set to 48 or 64 when compared with RCAN or RRDB.

      , "dataloader_shuffle": True
      , "dataloader_num_workers": 4
      , "dataloader_batch_size": 8      #// batch size 1 | 16 | 32 | 48 | 64 | 128. Total batch size =4x8=32 in SwinIR
    }
    , "test": {
      "name": "test_dataset"            #// just name
      , "dataset_type": "sr"         #// "dncnn" | "dnpatch" | "fdncnn" | "ffdnet" | "sr" | "srmd" | "dpsr" | "plain" | "plainpatch" | "jpeg"
      , "dataroot_H": "testsets/testH"  #// path of H testing dataset
      , "dataroot_L": "testsets/testL"              #// path of L testing dataset

    }
  }

  , "netG": {
    "net_type": "swinir"
    , "upscale": 3                      #// 2 | 3  | 4 | 8
    , "in_chans": 3
    , "img_size": 64                    #// For fair comparison, LR patch size is set to 48 or 64 when compared with RCAN or RRDB.
    , "window_size": 8
    , "img_range": 255
    , "depths": [6, 6, 6, 6, 6, 6]
    , "embed_dim": 180
    , "num_heads": [6, 6, 6, 6, 6, 6]
    , "mlp_ratio": 2
    , "upsampler": "pixelshuffle"        #// "pixelshuffle" | "pixelshuffledirect" | "nearest+conv" | null
    , "resi_connection": "1conv"        #// "1conv" | "3conv"

    , "init_type": "default"
  }

  , "train": {
    "G_lossfn_type": "l1"               #// "l1" preferred | "l2sum" | "l2" | "ssim" | "charbonnier"
    , "G_lossfn_weight": 1.0            #// default

    , "E_decay": 0.999                  #// Exponential Moving Average for netG: set 0 to disable; default setting 0.999

    , "G_optimizer_type": "adam"        #// fixed, adam is enough
    , "G_optimizer_lr": 2e-4            #// learning rate
    , "G_optimizer_wd": 0               #// weight decay, default 0
    , "G_optimizer_clipgrad": None      #// unused
    , "G_optimizer_reuse": True         #//

    , "G_scheduler_type": "MultiStepLR" #// "MultiStepLR" is enough
    , "G_scheduler_milestones": [250000, 400000, 450000, 475000, 500000]
    , "G_scheduler_gamma": 0.5

    , "G_regularizer_orthstep": None    #// unused
    , "G_regularizer_clipstep": None    #// unused

    , "G_param_strict": True
    , "E_param_strict": True

    , "checkpoint_test": 5000           #// for testing
    , "checkpoint_save": 5000           #// for saving model
    , "checkpoint_print": 200           #// for print
  }
}

with open("options/swinir/swinir_sr_x3_finetune.json", "w") as f:
    json.dump(config, f, indent=2)

print("JSON creado en options/swinir/")

JSON creado en options/swinir/


## Guardar repositorio con dataset y configuración de entrenamiento

In [ ]:
#Entrenamiento 1
#!cp -r /content/KAIR /content/drive/MyDrive/DOWNSCALING_2/Repositorio_TRAIN_1/KAIR_UPSAMPLE

#Entrenamiento 2
!cp -r /content/drive/MyDrive/DOWNSCALING_2/Repositorio_TRAIN_2/KAIR_UPSAMPLE_LAYER5

## Ejecución del entrenamiento

In [ ]:
!python main_train_psnr.py --opt options/swinir/swinir_sr_x3_finetune.json

#Reanudar Entrenamiento a partir de un repositorio guardado en Drive

## Cargar el repositorio

In [ ]:
#REPOSITORIO ENTRENAMIENTO 1
#!cp -r /content/drive/MyDrive/DOWNSCALING_2/Repositorio_TRAIN_1/KAIR_UPSAMPLE /content/KAIR

#REPOSITORIO ENTRENAMIENTO 2
!cp -r /content/drive/MyDrive/DOWNSCALING_2/Repositorio_TRAIN_2/KAIR_UPSAMPLE_LAYER5 /content/KAIR


%cd KAIR
!pip install -r requirement.txt

## Ejecución del entrenamiento

In [ ]:
!python main_train_psnr.py --opt options/swinir/swinir_sr_x3_finetune.json